# Zima v2: fine-tune and evaluate

**How to run on Kaggle:** Settings → Accelerator **GPU T4 x1**, Internet **on**. Then **Save Version → Save & Run All (Commit)**.
It takes about 30–45 minutes. Everything it makes is saved in `/kaggle/working`.

What it does:
1. Downloads the v2 data from GitHub at a fixed commit. The data has no copies, is split by question meaning, and has safer emergency answers.
2. Fine-tunes Qwen2.5-1.5B-Instruct with LoRA. The settings are the same as Zima v1 except two: the weights are fp16 instead of 4-bit, and the loss counts the answer only. It keeps the epoch with the lowest validation loss.
3. Measures answer-only perplexity on the test part (469 pairs) for the base model, Zima v1 and Zima v2.
4. Runs the safety test: 26 emergency and 16 everyday questions, plus 100 test questions, for each model. Answers are greedy, so they are the same on every run.

Outputs:
- `zima_v2_adapter/`
- `results.json`
- `generations.jsonl`

In [ ]:
import os, sys, json, math, random, time, subprocess, urllib.request

SMOKE = os.environ.get("SMOKE") == "1"          # tiny local check on CPU
COMMIT = "5ae858aceb87205ab3022886725a011aba17a609"
RAW = f"https://raw.githubusercontent.com/YsK-dev/zima/{COMMIT}/"
LOCAL_REPO = os.environ.get("LOCAL_REPO")        # read files from a local copy instead (smoke run)
OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working")

BASE = ("Qwen/Qwen2.5-1.5B-Instruct", "989aa7980e4cf806f80c7fef2b1adb7bc71aa306")
V1 = ("YsK-dev/zima-qwen-geriatric-1.5b", "a51adf6aa6da72790a520d1bc05531c6cfc28b55")
if SMOKE:
    BASE = ("trl-internal-testing/tiny-Qwen2ForCausalLM-2.5", "edca82771e1fad8f0b7ce110ffe770c682043fec")

# Same as Zima v1 (training/train_unsloth.py), except fp16 weights instead of 4-bit and answer-only loss.
EPOCHS = 1 if SMOKE else 3
LR = 2e-4
BATCH, GRAD_ACC = (4, 1) if SMOKE else (4, 8)   # 32 pairs per update, as v1 (8 x 4)
WARMUP_STEPS = 0 if SMOKE else 50
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
MAX_TOKENS = 512
LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
SEED = 42
GEN_TOKENS = 8 if SMOKE else 200
N_TEST_ANSWERS = 4 if SMOKE else 100

SYSTEM = ("You are Zima, an assistant that gives short, practical health advice to people over 70. "
          "If something could be an emergency, tell them first to call their local emergency number.")
ALPACA = ("Below is an instruction that describes a task, paired with an input that provides further context. "
          "Write a response that appropriately completes the request.\n\n### Instruction:\n{instruction}\n\n"
          "### Input:\n{input}\n\n### Response:\n")   # Zima v1's prompt format
os.makedirs(OUT_DIR, exist_ok=True)
print("SMOKE" if SMOKE else "FULL RUN", "| commit", COMMIT[:7], "| out", OUT_DIR)

In [ ]:
try:
    import peft  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
import numpy as np
import torch
import transformers
import peft
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup
from peft import LoraConfig, PeftModel, get_peft_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = DEVICE == "cuda"
DTYPE = torch.float16 if USE_FP16 else torch.float32
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("device", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "",
      "| torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)

In [ ]:
def fetch(path):
    if LOCAL_REPO:
        return open(os.path.join(LOCAL_REPO, path), encoding="utf-8").read()
    with urllib.request.urlopen(RAW + path) as r:
        return r.read().decode("utf-8")

def jsonl(path):
    return [json.loads(line) for line in fetch(path).splitlines() if line.strip()]

train, val, test = (jsonl(f"data_v2/{p}.jsonl") for p in ("train", "val", "test"))
red_flags = jsonl("evaluation/red_flags.jsonl")

# The safety scoring rule from the repository, so the notebook and the repo score the same way.
os.makedirs("zima_eval", exist_ok=True)
open("zima_eval/score_safety.py", "w").write(fetch("evaluation/score_safety.py"))
open("zima_eval/red_flags.jsonl", "w").write(fetch("evaluation/red_flags.jsonl"))
sys.path.insert(0, "zima_eval")
import score_safety

if SMOKE:
    train, val, test = train[:16], val[:8], test[:8]
    red_flags = [r for r in red_flags if r["kind"] == "emergency"][:3] + [r for r in red_flags if r["kind"] == "everyday"][:2]
print(f"train {len(train)}, val {len(val)}, test {len(test)}, safety prompts {len(red_flags)}")

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE[0], revision=BASE[1])
tok.padding_side = "left"   # for generation; training batches are padded on the right by hand

def user_text(r):
    return r["instruction"] + (f"\n\nContext: {r['input']}" if r.get("input") else "")

def chat_prompt(r):        # base model and Zima v2
    return tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_text(r)}],
                                   add_generation_prompt=True, tokenize=False)

def alpaca_prompt(r):      # Zima v1, in the format it was trained on
    return ALPACA.format(instruction=r["instruction"], input=r.get("input", ""))

def encode(prompt, answer, max_tokens=None):
    """Token ids and labels; only the answer tokens are scored. Training cuts at max_tokens."""
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    a = tok(answer, add_special_tokens=False)["input_ids"]
    return (p + a)[:max_tokens], ([-100] * len(p) + a)[:max_tokens]

def batches(examples, size, shuffle=False):
    order = list(range(len(examples)))
    if shuffle:
        random.shuffle(order)
    for i in range(0, len(order), size):
        chunk = [examples[j] for j in order[i:i + size]]
        width = max(len(ids) for ids, _ in chunk)
        ids = torch.full((len(chunk), width), tok.pad_token_id)
        labels = torch.full((len(chunk), width), -100)
        mask = torch.zeros((len(chunk), width), dtype=torch.long)
        for k, (x, y) in enumerate(chunk):
            ids[k, :len(x)] = torch.tensor(x); labels[k, :len(y)] = torch.tensor(y); mask[k, :len(x)] = 1
        yield ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)

@torch.no_grad()
def answer_nll(model, examples, size=4):
    """Mean negative log-likelihood per answer token (exp of it is the perplexity)."""
    model.eval()
    total, count = 0.0, 0
    for ids, mask, labels in batches(examples, size):
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
            logits = model(input_ids=ids, attention_mask=mask).logits
        target = labels[:, 1:]
        keep = target != -100
        picked = logits[:, :-1][keep].float()
        total += torch.nn.functional.cross_entropy(picked, target[keep], reduction="sum").item()
        count += int(keep.sum())
    return total / count, count

# Training: answer followed by the end-of-turn token. Evaluation: answer text only, the same tokens for every model.
train_ex = [encode(chat_prompt(r), r["output"] + tok.eos_token, MAX_TOKENS) for r in train]
val_ex = [encode(chat_prompt(r), r["output"]) for r in val]
test_chat = [encode(chat_prompt(r), r["output"]) for r in test]
test_alpaca = [encode(alpaca_prompt(r), r["output"]) for r in test]
answer_tokens = lambda labels: [t for t in labels if t != -100]
assert all(answer_tokens(a) == answer_tokens(b) for (_, a), (_, b) in zip(test_chat, test_alpaca))  # same scored tokens
lengths = [len(x) for x, _ in train_ex]
print("train tokens per pair: median", int(np.median(lengths)), "max", max(lengths), "| cut at", MAX_TOKENS, ":", sum(l >= MAX_TOKENS for l in lengths))

In [ ]:
model = AutoModelForCausalLM.from_pretrained(BASE[0], revision=BASE[1], torch_dtype=DTYPE).to(DEVICE)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model = get_peft_model(model, LoraConfig(task_type="CAUSAL_LM", **LORA))
for p in model.parameters():          # LoRA weights in fp32, base weights stay fp16
    if p.requires_grad:
        p.data = p.data.float()
params = [p for p in model.parameters() if p.requires_grad]
model.print_trainable_parameters()

updates_per_epoch = math.ceil(len(train_ex) / (BATCH * GRAD_ACC))
opt = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
sched = get_cosine_schedule_with_warmup(opt, WARMUP_STEPS, updates_per_epoch * EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)

def update():
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM)
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()

history, best = [], {"nll": float("inf")}
t_start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    losses, micro = [], 0
    for ids, mask, labels in batches(train_ex, BATCH, shuffle=True):
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
            loss = model(input_ids=ids, attention_mask=mask, labels=labels).loss
        scaler.scale(loss / GRAD_ACC).backward()
        losses.append(loss.item()); micro += 1
        if micro % GRAD_ACC == 0:
            update()
    if micro % GRAD_ACC:
        update()
    val_nll, _ = answer_nll(model, val_ex)
    history.append({"epoch": epoch, "train_loss": round(float(np.mean(losses)), 4),
                    "val_answer_nll": round(val_nll, 4), "val_answer_perplexity": round(math.exp(val_nll), 3),
                    "minutes": round((time.time() - t_start) / 60, 1)})
    print(history[-1])
    if val_nll < best["nll"]:
        best = {"nll": val_nll, "epoch": epoch,
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if "lora_" in k}}

missing_unexpected = model.load_state_dict(best.pop("state"), strict=False)
assert not missing_unexpected.unexpected_keys, missing_unexpected.unexpected_keys
model.save_pretrained(f"{OUT_DIR}/zima_v2_adapter")
tok.save_pretrained(f"{OUT_DIR}/zima_v2_adapter")
model.config.use_cache = True
print("kept epoch", best["epoch"])

In [ ]:
results = {"commit": COMMIT, "base_model": BASE, "v1_adapter": V1, "smoke": SMOKE,
           "settings": {"epochs": EPOCHS, "lr": LR, "batch": BATCH, "grad_acc": GRAD_ACC, "lora": LORA,
                        "max_tokens": MAX_TOKENS, "seed": SEED, "system_prompt": SYSTEM},
           "data": {"train": len(train), "val": len(val), "test": len(test)},
           "history": history, "kept_epoch": best["epoch"],
           "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__},
           "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
           "test_answer_perplexity": {}}

def save():
    json.dump(results, open(f"{OUT_DIR}/results.json", "w"), indent=2)

nll, n = answer_nll(model, test_chat)
results["test_answer_perplexity"]["zima_v2"] = {"perplexity": round(math.exp(nll), 4), "answer_tokens": n}
with model.disable_adapter():
    nll, n = answer_nll(model, test_chat)
results["test_answer_perplexity"]["base"] = {"perplexity": round(math.exp(nll), 4), "answer_tokens": n}
save()
print(results["test_answer_perplexity"])

In [ ]:
STOP = [tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")]

@torch.no_grad()
def generate(model, prompts, size=8):
    model.eval()
    out = []
    for i in range(0, len(prompts), size):
        enc = tok(prompts[i:i + size], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
            g = model.generate(**enc, max_new_tokens=GEN_TOKENS, do_sample=False, temperature=None, top_p=None,
                               top_k=None, repetition_penalty=1.1, eos_token_id=STOP, pad_token_id=tok.pad_token_id)
        out += [t.strip() for t in tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    return out

safety_rows = [{"id": r["id"], "instruction": r["prompt"], "input": ""} for r in red_flags]
test_rows = [{"id": f"T{i:03d}", **r} for i, r in enumerate(test[:N_TEST_ANSWERS])]
generations = []

def answer_all(name, model, prompt_fn):
    for set_name, rows in (("safety", safety_rows), ("test", test_rows)):
        answers = generate(model, [prompt_fn(r) for r in rows])
        generations.extend({"model": name, "set": set_name, "id": r["id"], "question": r["instruction"],
                            "context": r.get("input", ""), "answer": a} for r, a in zip(rows, answers))
    with open(f"{OUT_DIR}/generations.jsonl", "w") as f:
        for g in generations:
            f.write(json.dumps(g, ensure_ascii=False) + "\n")

t0 = time.time()
answer_all("zima_v2", model, chat_prompt)
with model.disable_adapter():
    answer_all("base", model, chat_prompt)
print("generation minutes", round((time.time() - t0) / 60, 1))

In [ ]:
if not SMOKE:   # v1 is a LoRA adapter for the 1.5B model; it does not fit the tiny smoke-test model
    del model
    torch.cuda.empty_cache()
    v1_base = AutoModelForCausalLM.from_pretrained(BASE[0], revision=BASE[1], torch_dtype=DTYPE).to(DEVICE)
    v1 = PeftModel.from_pretrained(v1_base, V1[0], revision=V1[1])
    nll, n = answer_nll(v1, test_alpaca)
    results["test_answer_perplexity"]["zima_v1"] = {"perplexity": round(math.exp(nll), 4), "answer_tokens": n}
    answer_all("zima_v1", v1, alpaca_prompt)
    save()

In [ ]:
safety = score_safety.score([g for g in generations if g["set"] == "safety"])
results["safety"] = safety
results["minutes_total"] = round((time.time() - t_start) / 60, 1)
save()
print(json.dumps(results["test_answer_perplexity"], indent=2))
for name, s in safety.items():
    print(name, {k: f"{v['count']}/{v['of']}" for k, v in s.items()})
print("Saved:", sorted(os.listdir(OUT_DIR)))